# Chicago Restaurant Inspection Failure Classification

## Load and prepare the data:


In [3]:
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, precision_score, recall_score, f1_score

inspections = pd.read_csv("data/raw/Food_Inspections_20261012.csv")
licenses = pd.read_csv("data/raw/Business_Licenses_20261105.csv", low_memory=False)
cca = pd.read_csv("data/raw/CommunityDataSnapshots_Historical_455993172616713182.csv")
buildings = pd.read_csv("data/raw/Building_Footprints_20261105.csv", low_memory=False)

# Select relevant columns.
inspections = inspections[[
    "Inspection ID", "DBA Name", "License #", "Facility Type",
    "Address", "Inspection Date", "Results", "Violations", "Risk",
    "Inspection Type"
]].copy()

licenses = licenses[[
    "LICENSE NUMBER", "COMMUNITY AREA"
]].copy()

cca = cca[[
    "GEOID", "GEOG", "TOT_POP", "MEDINC"
]].copy()

buildings = buildings[[
    "BLDG_ID", "F_ADD1", "T_ADD1", "PRE_DIR1",
    "ST_NAME1", "ST_TYPE1", "SUF_DIR1", "YEAR_BUILT"
]].copy()

# Keep restaurant inspections from all available years.
inspections["Inspection Date"] = pd.to_datetime(
    inspections["Inspection Date"]
)

inspections = inspections[
    inspections["Facility Type"] == "Restaurant"
].copy()

inspections = inspections.drop_duplicates()

# Standardize matching columns.
inspections["License #"] = pd.to_numeric(inspections["License #"])
licenses["LICENSE NUMBER"] = pd.to_numeric(licenses["LICENSE NUMBER"])
licenses["COMMUNITY AREA"] = pd.to_numeric(licenses["COMMUNITY AREA"])
cca["GEOID"] = pd.to_numeric(cca["GEOID"])
cca["MEDINC"] = pd.to_numeric(cca["MEDINC"])

# Collapse repeated license-to-area mappings, including renewals.
licenses = licenses.dropna().drop_duplicates()
licenses = licenses[licenses["LICENSE NUMBER"] > 0]

# Exclude licenses with conflicting community-area mappings.
licenses = licenses[
    ~licenses["LICENSE NUMBER"].duplicated(keep=False)
]

cca = cca.dropna(subset=["GEOID"])

# Join inspections to licenses.
combined = inspections.merge(
    licenses,
    left_on="License #",
    right_on="LICENSE NUMBER",
    how="inner",
    validate="many_to_one"
)

# Add community-area demographics.
combined = combined.merge(
    cca,
    left_on="COMMUNITY AREA",
    right_on="GEOID",
    how="inner",
    validate="many_to_one"
)

# Clean building street numbers and construction years.
for col in ["F_ADD1", "T_ADD1", "YEAR_BUILT"]:
    buildings[col] = pd.to_numeric(
        buildings[col]
        .astype("string")
        .str.replace(",", "", regex=False)
    )

# First pass: single-number addresses with usable construction years.
buildings = buildings[
    buildings["YEAR_BUILT"].between(1, 2026)
    & buildings["F_ADD1"].notna()
    & (buildings["F_ADD1"] == buildings["T_ADD1"])
].copy()

# Assemble building addresses.
buildings["Address_Match"] = (
    buildings["F_ADD1"].astype("Int64").astype("string")
    + " " + buildings["PRE_DIR1"].fillna("")
    + " " + buildings["ST_NAME1"].fillna("")
    + " " + buildings["ST_TYPE1"].fillna("")
    + " " + buildings["SUF_DIR1"].fillna("")
).str.upper().str.replace(r"\s+", " ", regex=True).str.strip()

# Standardize inspection addresses.
combined["Address_Match"] = (
    combined["Address"]
    .str.upper()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Keep addresses identifying a single building.
building_lookup = buildings[[
    "Address_Match", "BLDG_ID", "YEAR_BUILT"
]].drop_duplicates()

building_lookup = building_lookup[
    ~building_lookup["Address_Match"].duplicated(keep=False)
]

# Add building data without dropping unmatched inspections.
combined = combined.merge(
    building_lookup,
    on="Address_Match",
    how="left",
    validate="many_to_one"
)

# A building constructed after an inspection is not a valid historical match.
future_building = (
    combined["YEAR_BUILT"] > combined["Inspection Date"].dt.year
).fillna(False)

combined.loc[
    future_building, ["BLDG_ID", "YEAR_BUILT"]
] = pd.NA

# Calculate building age at the time of inspection.
combined["Building_Age"] = (
    combined["Inspection Date"].dt.year - combined["YEAR_BUILT"]
)

# Keep only exact Pass and Fail outcomes for the comparison.
results = combined[
    combined["Results"].isin(["Pass", "Fail"])
].copy()

# Summarize outcomes by community area.
summary = (
    results.groupby(["GEOG", "MEDINC", "Results"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["Pass", "Fail"], fill_value=0)
    .reset_index()
)

summary["Total"] = summary["Pass"] + summary["Fail"]
summary["Fail_Percent"] = summary["Fail"] / summary["Total"] * 100
summary["Pass_Fail_Gap"] = summary["Pass"] - summary["Fail"]

summary = summary.sort_values("Fail_Percent", ascending=False)

# Check the results.
print("Restaurant inspections before joins:", len(inspections))
print("Inspections matched to licenses and CMAP:", len(combined))
print(
    "Inspections with building age:",
    combined["Building_Age"].notna().sum()
)
print(
    "Inspection date range:",
    combined["Inspection Date"].min(),
    "to",
    combined["Inspection Date"].max()
)

display(combined[[
    "DBA Name", "Address", "Inspection Date", "GEOG",
    "Results", "YEAR_BUILT", "Building_Age"
]].head(10))

display(summary)


Restaurant inspections before joins: 215026
Inspections matched to licenses and CMAP: 211493
Inspections with building age: 58084
Inspection date range: 2010-01-04 00:00:00 to 2026-10-09 00:00:00


,DBA Name,Address,Inspection Date,GEOG,Results,YEAR_BUILT,Building_Age
0,SIMPLY SOUPS SALADS & SANDWICHES,635 E 47TH ST,2026-10-09,Grand Boulevard,Pass,2008,18
1,AGLAIA COFFEE AND TEA COMPANY,925 W CHICAGO AVE,2026-10-09,West Town,Fail,1904,122
2,SIP & SAVOR COFFEE HOUSE,528 E 43RD ST,2026-10-09,Grand Boulevard,Fail,<NA>,<NA>
3,FUERZA VITAL,8542 S PULASKI RD,2026-10-09,Ashburn,Pass,<NA>,<NA>
4,SHARKS ON DIVISION CORPORATION,5751 W DIVISION ST,2026-10-08,Austin,Fail,<NA>,<NA>
5,LA QUINTA INN AND SUITES CHICAGO DOWNTOWN,1 S FRANKLIN ST,2026-10-08,The Loop,Pass,1958,68
6,CILANTRO TACO GRILL - LOOP,180 N WELLS ST,2026-10-08,The Loop,Pass,<NA>,<NA>
7,LA BOMBA FOOD,3221 W ARMITAGE AVE,2026-10-08,Logan Square,Fail,<NA>,<NA>
8,THALI CORNER,2350 W DEVON AVE,2026-10-08,West Ridge,Out of Business,<NA>,<NA>
9,ADORO FUSION KITCHEN + BAR,2509 W NORTH AVE,2026-10-08,West Town,Fail,<NA>,<NA>


Results,GEOG,MEDINC,Pass,Fail,Total,Fail_Percent,Pass_Fail_Gap
67,Washington Park,35737.179487,188,125,313,39.936102,63
57,Riverdale,17638.888889,17,10,27,37.037037,7
70,West Garfield Park,38641.732283,330,175,505,34.653465,155
18,East Garfield Park,37170.329670,246,126,372,33.870968,120
28,Greater Grand Crossing,37961.582569,866,442,1308,33.792049,424
...,...,...,...,...,...,...,...
3,Ashburn,82235.360360,860,202,1062,19.020716,658
15,Clearing,78436.888454,580,116,696,16.666667,464
26,Garfield Ridge,96218.190386,1646,328,1974,16.616008,1318
53,O'Hare,83115.610712,2597,509,3106,16.387637,2088


## Feature:

In [4]:
results["year"] = results["Inspection Date"].dt.year
results["month"] = results["Inspection Date"].dt.month

cat = ["Risk", "Inspection Type", "GEOG"]
num = ["year", "month", "Building_Age"]

features = results[cat + num].copy()
target = results["Results"].eq("Fail").astype(int)

print("Feature matrix shape:", features.shape)
display(target.value_counts().rename(index={0: "Pass (0)", 1: "Fail (1)"}).to_frame("count"))

Feature matrix shape: (147309, 6)


,count
Results,
Pass (0),108053
Fail (1),39256


## Classification:

In [5]:
prep = ColumnTransformer([
    (
        'numeric',
        Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scale', StandardScaler())
        ]),
        num
    ),
    (
        'categorical',
        Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ]),
        cat
    )
])

def learn_classifier():
    return Pipeline([
        ('prep', prep),
        ('model', LogisticRegression(
            max_iter=1000,
            class_weight='balanced'
        ))
    ])

baseline = DummyClassifier(strategy='most_frequent')
classifier = learn_classifier()

print(baseline)
print(classifier)

DummyClassifier(strategy='most_frequent')
Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scale',
                                                                   StandardScaler())]),
                                                  ['year', 'month',
                                                   'Building_Age']),
                                                 ('categorical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                              

## Evaluation:

In [6]:
def evaluate_classifier(model, X, y, cv, scoring):
    results = cross_validate(
        model, X, y,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )
    return {
        metric: results[f'test_{metric}'].mean()
        for metric in scoring
    }


kf = StratifiedKFold(
    n_splits=4,
    shuffle=True,
    random_state=1
)

scoring = {
    'Accuracy': 'accuracy',
    'Fail precision': make_scorer(precision_score, zero_division=0),
    'Fail recall': make_scorer(recall_score, zero_division=0),
    'Fail F1-score': make_scorer(f1_score, zero_division=0)
}


baseline_scores = evaluate_classifier(baseline, features, target, kf, scoring)
classifier_scores = evaluate_classifier(classifier, features, target, kf, scoring)

scores = pd.DataFrame(
    [baseline_scores, classifier_scores],
    index=['Majority baseline', 'Logistic regression']
)

display(scores.style.format('{:.3f}'))

,Accuracy,Fail precision,Fail recall,Fail F1-score
Majority baseline,0.734,0.000,0.000,0.000
Logistic regression,0.599,0.372,0.735,0.494


## Analysis:
The baseline achieved 73.4% accuracy by predicting that every inspection would pass. It did not identify any failed inspections. Logistic regression had lower accuracy at 59.9%, but it identified 73.5% of actual failures. Of the inspections it predicted would fail, 37.2% actually failed, resulting in a Fail F1-score of 49.4%.

This suggests logistic regression is better at finding failed inspections, but it also produces many false alarms. There is still substantial room to improve the model. We believe it is limited by the information available in our dataset. Building age is missing for many inspections, and our features may not capture important restaurant details, such as food type or past inspection history. Adding more complete data and useful features may improve its predictions.